In [14]:
print("hello residual")

hello residual


In [15]:
!pip install kaggle==1.5.16

In [16]:
! kaggle competitions download challenges-in-representation-learning-facial-expression-recognition-challenge

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [17]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge.zip

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
replace example_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [18]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam, SGD, RMSprop # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory

device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cpu


In [19]:
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

In [20]:
x_train = np.array([
    list(map(int, row.split()))
    for row in train_df['pixels']
]).reshape(-1, 48, 48)

y_train = train_df['emotion'].values

In [21]:
from sklearn.model_selection import train_test_split

x_train = x_train / 255.0
x_train = x_train.reshape(-1, 1, 48, 48)

X_train, X_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)

Train shape: (22967, 1, 48, 48)
Val shape: (5742, 1, 48, 48)


In [22]:
class FERDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_dataset = FERDataset(X_train, y_train)
val_dataset   = FERDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [23]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate=0.25):
        super(ResidualBlock, self).__init__()

        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
        )

        # Shortcut connection — if channels change, use 1x1 conv to match dimensions
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout2d(dropout_rate)

    def forward(self, x):
        residual = self.shortcut(x)   # ← skip connection
        out = self.conv_block(x)
        out = out + residual          # ← add input back to output
        out = self.relu(out)
        out = self.dropout(out)
        return out

In [24]:
class ResNetCNN(nn.Module):
    def __init__(self, dropout_rate=0.25):
        super(ResNetCNN, self).__init__()

        # Initial conv to expand channels before residual blocks
        self.initial = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        # Block 1: 64 → 64 | output: (64, 24, 24)
        self.block1 = nn.Sequential(
            ResidualBlock(64, 64, dropout_rate),
            nn.MaxPool2d(2, 2)
        )

        # Block 2: 64 → 128 | output: (128, 12, 12)
        self.block2 = nn.Sequential(
            ResidualBlock(64, 128, dropout_rate),
            nn.MaxPool2d(2, 2)
        )

        # Block 3: 128 → 256 | output: (256, 6, 6)
        self.block3 = nn.Sequential(
            ResidualBlock(128, 256, dropout_rate),
            ResidualBlock(256, 256, dropout_rate),  # ← extra residual same channels
            nn.MaxPool2d(2, 2)
        )

        # Block 4: 256 → 512 | output: (512, 3, 3)
        self.block4 = nn.Sequential(
            ResidualBlock(256, 512, dropout_rate),
            nn.MaxPool2d(2, 2)
        )

        # Classifier
        self.fc = nn.Sequential(
            nn.Flatten(),               # 512*3*3 = 4608
            nn.Linear(512 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 7)
        )

    def forward(self, x):
        x = self.initial(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.fc(x)
        return x

# Sanity check
model = ResNetCNN().to(device)
print(model)

dummy = torch.randn(1, 1, 48, 48).to(device)
print("Output shape:", model(dummy).shape)  # should be torch.Size([1, 7])

ResNetCNN(
  (initial): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (block1): Sequential(
    (0): ResidualBlock(
      (conv_block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (shortcut): Identity()
      (relu): ReLU()
      (dropout): Dropout2d(p=0.25, inplace=False)
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): ResidualBlock(
      (conv_block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride

In [28]:
from torchvision import transforms
from torch.utils.data import Dataset

class FERDatasetAugmented(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = images
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.augment = augment

        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.RandomAffine(
                degrees=0,
                translate=(0.1, 0.1)   # slight shift
            ),
        ]) if augment else None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        if self.augment and self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Augmented dataset only for training, not validation
train_dataset_aug = FERDatasetAugmented(X_train, y_train, augment=True)
val_dataset_aug   = FERDatasetAugmented(X_val, y_val, augment=False)

In [25]:
!pip install wandb -q
import wandb
wandb.login()  # will prompt for API key

True

In [29]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

experiments = [
    {
        "name": "resnet-cnn-adam-lr0.001-drop0.25",
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "batch_size": 64,
        "epochs": 50,
        "dropout_rate": 0.25,
        "augment": True,
    },
    {
        "name": "resnet-cnn-adam-lr0.0001-drop0.25",
        "learning_rate": 0.0001,
        "optimizer": "Adam",
        "batch_size": 64,
        "epochs": 50,
        "dropout_rate": 0.25,
        "augment": True,
    },
    {
        "name": "resnet-cnn-adam-lr0.001-drop0.5",
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "batch_size": 64,
        "epochs": 50,
        "dropout_rate": 0.5,
        "augment": True,
    },
    {
        "name": "resnet-cnn-adam-batch32-drop0.25",
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "batch_size": 32,
        "epochs": 50,
        "dropout_rate": 0.25,
        "augment": True,
    },
    {
        "name": "resnet-cnn-adam-noaug-drop0.25",
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "batch_size": 64,
        "epochs": 50,
        "dropout_rate": 0.25,
        "augment": False,              # compare with/without augmentation
    },
]

def get_optimizer(name, model, lr):
    if name == "Adam":
        return Adam(model.parameters(), lr=lr)
    elif name == "SGD":
        return SGD(model.parameters(), lr=lr, momentum=0.9)
    elif name == "RMSprop":
        return RMSprop(model.parameters(), lr=lr)

def run_experiment(config):
    run = wandb.init(
        project="Facial_Expression_Recognition",
        name=config["name"],
        group="resnet-cnn",            # ← separate group from deep-cnn
        tags=["resnet-cnn", "residual", "batchnorm", "dropout", config["optimizer"].lower()],
        config={
            **config,
            "architecture": "ResNetCNN",
            "num_conv_blocks": 4,
            "conv_filters": [64, 128, 256, 512],
            "fc_units": [1024, 512, 128],
            "residual_connections": True,
            "batch_norm": True,
            "dataset": "FER2013"
        }
    )

    model = ResNetCNN(dropout_rate=config["dropout_rate"]).to(device)
    optimizer = get_optimizer(config["optimizer"], model, config["learning_rate"])
    criterion = nn.CrossEntropyLoss()

    scheduler = ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    train_dataset_run = FERDatasetAugmented(X_train, y_train, augment=config["augment"])
    val_dataset_run   = FERDatasetAugmented(X_val, y_val, augment=False)

    train_loader = DataLoader(train_dataset_run, batch_size=config["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_dataset_run, batch_size=config["batch_size"], shuffle=False)

    wandb.watch(model, log="all", log_freq=10)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(config["epochs"]):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc  = correct / total

        # --- Validate ---
        model.eval()
        total_loss, correct, total = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                total_loss += loss.item()
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)

        val_loss = total_loss / len(val_loader)
        val_acc  = correct / total

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
            "learning_rate": current_lr,
        })

        print(f"[{config['name']}] Epoch {epoch+1}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"LR: {current_lr:.6f}")

    # --- Plot & Log ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(config["name"])

    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(train_accs, label='Train Acc')
    ax2.plot(val_accs, label='Val Acc')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    wandb.log({"training_curves": wandb.Image(fig)})
    plt.show()
    plt.close(fig)

    wandb.summary["best_val_accuracy"]    = max(val_accs)
    wandb.summary["best_val_loss"]        = min(val_losses)
    wandb.summary["final_train_accuracy"] = train_accs[-1]
    wandb.summary["overfit_gap"]          = train_accs[-1] - val_accs[-1]

    wandb.finish()
    print(f"\n✅ Finished: {config['name']}\n{'='*60}\n")

for config in experiments:
    run_experiment(config)

KeyboardInterrupt: 